# Notebook 6: simulation with the full PE3 process (response to Dr. Gaur's review)
**Author:** Aly Gallo · ChroniCare-AI protocol v0.5.1 · Continues notebooks 1 to 5.

> **Note on this English version (19 November 2026).** This is a translation of the original Spanish notebook (kept in the `es/` folder with its saved outputs). No formulas, seeds or logic were changed; only comments, explanations and printed messages were translated. Variable names and CSV column names remain in Spanish (see the glossary in the README). This notebook is the simulation written for protocol v0.5.1 (one-sided alpha 0.05). The figures of protocol v0.6 (one-sided alpha 0.025, Section 6.4.7) come from Simulation 7 (`run_sim7.py`), which uses the same generator and seeds. The saved outputs were not copied into this version: run it to regenerate them.

**What this notebook does.** It generates fake patients (the truth is known) and applies to them the full analysis of the protocol:
robust nadir → recovery R<sub>i</sub> → expected recovery with patient-level *cross-fitting* (without baseline BSI) → AC<sub>i</sub> = R<sub>i</sub> − Ê[R|S,X] → average slope β<sub>mean</sub> (one-sided) with cluster-robust (patient) standard error.
It responds to the review points on the primary test:
- **Same outcome and same process as the primary analysis** (before, a simple slope on the fraction was used).
- **≥ 1,000 null repetitions** (here 2,000 to calibrate and another 1,000 independent ones to verify the size of the test), with the **Monte Carlo error** of each size and power.
- **Calibration grid** of coupling (0 to 0.36) × noise (0.15; 0.30; 0.60). The critical value is the **largest** 95th percentile over the whole grid, for each N.
- **Expected null slope** next to each estimate (the bias produced by coupling and noise).
- **Power of the two tests** (average and ceiling contrast) with the same process.
- **Sufficiency criteria** (calibrated power ≥ 0.80 and CI half-width ≤ reference effect).

**Important:** everything is simulated. The generating model (noise, measurement pattern, shape of the recovery) consists of **assumptions** until the hospital's data dictionary is available; the definitive numbers are recomputed with real noise, frequency and coupling, estimated without looking at the outcome.

## Reproducibility box

In [ ]:
import sys, os, platform, datetime, json, time, hashlib, warnings
import numpy as np, pandas as pd
import statsmodels, scipy, joblib, matplotlib
warnings.simplefilter("ignore")

SEED  = 20261113        # master seed; each grid cell uses a derived, fixed seed (reproducible)
QUICK = False           # True = quick test (a few minutes); False = full run

print("Date:", datetime.date.today(), "| Python", sys.version.split()[0], "|", platform.platform())
VERS = {"numpy": np.__version__, "pandas": pd.__version__, "statsmodels": statsmodels.__version__,
        "scipy": scipy.__version__, "joblib": joblib.__version__, "matplotlib": matplotlib.__version__}
print(VERS)
print("SEED =", SEED, "| QUICK =", QUICK, "| cores:", os.cpu_count())
print("DATA: none real. Everything is simulated. The generator parameters come from notebooks 4 and 5 (VitalDB cohort, n = 738;")
print("       estimated coupling 0.30 to 0.34, upper limit 0.36; noise 0.15 and 0.32).")

## Step 1. The simulator (saved as `fullsim.py`)
What is inside, in simple words:
- Each patient has a **true baseline state** (BSI) and 1 to 3 events that share it.
- Each event makes the BSI drop and then it recovers. The **drop is deeper when the baseline BSI is higher** in proportion to the **coupling** `c` (0 = none).
- It is measured with **noise** (`noise`). The baseline is the average of 1 or more prior measurements.
- Nadir = mean of the 2 lowest measurements in days 0–4. Final value = mean of the last 2 in days 5–7.
- **Clinical inclusion (primary):** all events with enough measurements enter, with no BSI-based threshold (as Section 6.4.1 requires). `Rb` repeats the analysis with the observed-drop threshold > 0.5 SD of notebooks 4 and 5 (sensitivity). `F` is the fraction recovered R/D (candidate for the switching rule of v0.5.1).
- Worlds: **null** (no association), **linear** (+0.05 or +0.10 of recoverable fraction per SD of BSI) and **ceiling** (worse baseline state, lower maximum recoverable fraction).

In [ ]:
%%writefile fullsim.py
"""Simulation 6 (ChroniCare-AI, protocol v0.5.1): FULL PE3 pipeline on simulated patients.
Everything is simulated: no real data are used. The parameters come from notebooks 4 and 5 (VitalDB).
BSI in SD units (higher = better). Pipeline per event:
  robust nadir (mean of the 2 lowest values in the nadir window) -> R_i = end - nadir ->
  expected recovery with PATIENT-level cross-fitting (without baseline BSI) -> AC_i = R_i - E^[R|S,X] ->
  primary test: least-squares slope beta_mean of AC on baseline BSI (+ covariates), one-sided.
It is run in the same way with F_i = R_i / D_i (fraction of the drop recovered), for the switching rule of v0.5.1 (6.4.7; in v0.6 F is a preregistered secondary outcome).
"""
import warnings
import numpy as np
from statsmodels.regression.quantile_regression import QuantReg

warnings.filterwarnings('ignore')

# ---- fixed generator parameters (same as notebooks 4 and 5 except where noted) ----
W = 7.0          # recovery window (days)
T0 = 2.0         # day of the true nadir
TAU = 2.0        # time scale of the recovery
WN = 4.0         # nadir window: days 0 to WN
WE = 2.0         # closing window: last WE days
UMBRAL = 0.5     # minimum observed drop (SD of the BSI) to include the event
KMAX = 14        # maximum post-event measurements per event
K_FOLDS = 5      # patient-level cross-fitting


def generar(rng, n, noise, c, world="null", effect=0.0, techo=None, dens=4.0):
    """Generates n patients (1 to 3 events each, sharing their latent baseline state).
    world: 'null' (no association), 'linear' (the recoverable fraction rises by `effect` per SD of BSI) or 'ceiling'
    (ceiling: worse baseline state -> lower maximum recoverable fraction; `techo` = 0.5 moderate, 1.0 strong).
    c = coupling (how much deeper the drop is when the baseline BSI is high)."""
    age = rng.normal(size=n)
    theta = -0.3 * age + np.sqrt(1 - 0.09) * rng.normal(size=n)       # true baseline state (SD=1)
    ne = 1 + np.minimum(rng.poisson(0.4, n), 2)                         # events per patient
    pid = np.repeat(np.arange(n), ne); E = len(pid)
    th, agez = theta[pid], age[pid]
    S = rng.uniform(0, 1, E)                                            # clinical severity (independent of the BSI)
    D = (0.8 + 0.4 * S) * np.clip(1 + c * th, 0.3, None) * np.exp(0.1 * rng.normal(size=E))
    u = rng.beta(4, 2, E) * (1 - 0.25 * S)                              # the expected recovery depends on S
    if world == "linear":
        r = np.clip(u + effect * th, 0.02, 1.0)
    elif world == "ceiling":
        r = np.minimum(u, np.clip(0.95 + techo * th, 0.05, 1.0))
    else:
        r = np.clip(u, 0.02, 1.0)
    # post-event measurements (parametric times, denser in the most severe events)
    npost = np.minimum(3 + rng.poisson(dens * S), KMAX)
    t = np.sort(np.where(np.arange(KMAX)[None, :] < npost[:, None], rng.uniform(0.1, W, (E, KMAX)), np.inf), axis=1)
    valid = np.isfinite(t); tt = np.where(valid, t, 0.0)
    k = 1 - np.exp(-(W - T0) / TAU)
    rec = r[:, None] * (1 - np.exp(-(tt - T0) / TAU)) / k
    q = np.where(tt <= T0, tt / T0, 1 - rec)
    x = th[:, None] - D[:, None] * q
    obs = x + noise * rng.normal(size=x.shape)
    npre = 1 + rng.poisson(1.0, E)
    base = th + noise * rng.normal(size=E) / np.sqrt(npre)             # mean of the prior measurements
    # robust nadir: mean of the two lowest values within [0, WN]
    inwin = valid & (t <= WN)
    srt = np.sort(np.where(inwin, obs, np.inf), axis=1)[:, :2]
    n_nad = inwin.sum(1)
    nadir = srt.mean(1)
    # final value: mean of the (up to) two last measurements within [W-WE, W]
    inclo = valid & (t >= W - WE)
    n_clo = inclo.sum(1)
    idx = np.argsort(-np.where(inclo, t, -np.inf), axis=1)[:, :2]
    v2 = np.take_along_axis(obs, idx, 1)
    ok2 = np.take_along_axis(inclo, idx, 1)
    fin = np.where(ok2.sum(1) > 0, (v2 * ok2).sum(1) / np.maximum(ok2.sum(1), 1), np.nan)
    Dobs = base - nadir
    inc = (n_nad >= 2) & (n_clo >= 1)                                   # clinical inclusion: measurement availability only
    R = fin - nadir
    return dict(pid=pid[inc], base=base[inc], S=S[inc], agez=agez[inc], dens=n_nad[inc].astype(float),
                R=R[inc], D=Dobs[inc])


def _cross_fit(Y, Z, pid):
    """AC = Y - E^[Y|Z] with patient-level cross-fitting (all events of a patient fall in the same fold)."""
    fold = pid % K_FOLDS
    pred = np.empty_like(Y)
    for f in range(K_FOLDS):
        te = fold == f; tr = ~te
        b = np.linalg.lstsq(Z[tr], Y[tr], rcond=None)[0]
        pred[te] = Z[te] @ b
    return Y - pred


def _ols_cluster(X, y, pid):
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ b
    bread = np.linalg.inv(X.T @ X)
    starts = np.r_[0, np.flatnonzero(np.diff(pid)) + 1]
    sc = np.add.reduceat(X * e[:, None], starts, axis=0)
    G = len(starts)
    V = bread @ (sc.T @ sc) @ bread * G / max(G - 1, 1)
    return b, np.sqrt(V[1, 1])


def _un_analisis(d, y, quantile=False):
    pid = d["pid"]; one = np.ones(len(pid))
    Z = np.column_stack([one, d["S"], d["agez"], d["dens"]])           # E^[.|S,X]: WITHOUT baseline BSI
    X = np.column_stack([one, d["base"], d["S"], d["agez"], d["dens"]])  # primary model: WITH baseline BSI
    ac = _cross_fit(y, Z, pid)
    b, se = _ols_cluster(X, ac, pid)
    out = [float(b[1]), float(se)]
    if quantile:
        b90 = QuantReg(ac, X).fit(q=0.90, max_iter=1000, p_tol=1e-4).params[1]
        b50 = QuantReg(ac, X).fit(q=0.50, max_iter=1000, p_tol=1e-4).params[1]
        out.append(float(b90 - b50))
    return out


def analizar(d, quantile=False):
    """Full pipeline on a simulated data set. Three analyses:
    R  = absolute recovery, ALL events with enough measurements (clinical inclusion; the protocol's primary);
    Rb = same, but only events with observed drop > UMBRAL (BSI-based criterion, as in notebooks 4 and 5; sensitivity);
    F  = fraction of the drop recovered R/D, only events with D > UMBRAL (candidate for the switching rule of v0.5.1, 6.4.7)."""
    out = {"n_pac": int(len(np.unique(d["pid"]))), "n_ev": int(len(d["pid"])), "Dbar": float(d["D"].mean())}
    r = _un_analisis(d, d["R"], quantile)
    out["b_R"], out["se_R"] = r[0], r[1]
    if quantile:
        out["dbeta_R"] = r[2]
    k = d["D"] > UMBRAL
    dk = {key: v[k] for key, v in d.items()}
    out["n_pac_b"] = int(len(np.unique(dk["pid"]))); out["Dbar_b"] = float(dk["D"].mean())
    rb = _un_analisis(dk, dk["R"], False); out["b_Rb"], out["se_Rb"] = rb
    f = _un_analisis(dk, np.clip(dk["R"] / dk["D"], -0.5, 1.5), False); out["b_F"], out["se_F"] = f
    return out


def generar_elegibles(rng, n_el, noise, c, world="null", effect=0.0, techo=None):
    """Generates enough candidates to obtain EXACTLY n_el eligible patients (the first n_el,
    which are a random sample because the patients are generated in random order)."""
    n_cand = int(n_el / 0.55) + 100
    while True:
        d = generar(rng, n_cand, noise, c, world, effect, techo)
        up = np.unique(d["pid"])
        if len(up) >= n_el:
            keep = d["pid"] <= up[n_el - 1]
            return {k: v[keep] for k, v in d.items()}
        n_cand *= 2


def repetir(seed_seq, reps, n_el, noise, c, world="null", effect=0.0, techo=None, quantile=False):
    """Repeats reps times (own seed, reproducible). n_el = ELIGIBLE patients (after the inclusion rule).
    Returns a dict of arrays (one entry per repetition)."""
    rng = np.random.default_rng(seed_seq)
    res = [analizar(generar_elegibles(rng, n_el, noise, c, world, effect, techo), quantile) for _ in range(reps)]
    return {k: np.array([r[k] for r in res]) for k in res[0]}

## Step 2. Configuration and proof that it is reproducible

In [ ]:
if QUICK:
    N_EL = [500, 1000]; NOISES = [0.15, 0.60]; CS = [0.0, 0.36]
    REPS_CAL, REPS_VER, REPS_POW = 100, 100, 100
    N_CEIL, NOISES_CEIL, CS_CEIL, REPS_CEIL = [1000], [0.30], [0.36], 40
else:
    N_EL = [500, 738, 1000, 2000, 3000, 5000]      # ELIGIBLE patients (after the inclusion rule)
    NOISES = [0.15, 0.30, 0.60]                      # measurement noise (SD of the BSI)
    CS = [0.0, 0.12, 0.24, 0.36]                     # coupling (0.36 = upper limit observed in VitalDB)
    REPS_CAL, REPS_VER, REPS_POW = 2000, 1000, 1000  # calibration / size verification / power
    N_CEIL, NOISES_CEIL, CS_CEIL, REPS_CEIL = [738, 2000, 5000], [0.15, 0.30], [0.0, 0.36], 300
EFFECTS = [0.05, 0.10]     # linear effect: recoverable fraction per SD of BSI (0.10 = the protocol's reference effect)
TECHOS = [0.5, 1.0]        # moderate and strong ceiling
OUT = ["R", "F", "Rb"]     # R = primary (clinical inclusion); F = fraction; Rb = sensitivity with BSI threshold
POWER_TARGET = 0.80

import fullsim as FS
# Determinism test: same seed -> identical results
a = FS.repetir(np.random.SeedSequence(SEED, spawn_key=(9, 9, 9, 9)), 20, 500, 0.30, 0.24)
b = FS.repetir(np.random.SeedSequence(SEED, spawn_key=(9, 9, 9, 9)), 20, 500, 0.30, 0.24)
print("Same seed, two identical runs:", all(np.array_equal(a[k], b[k]) for k in a))
# Seed sensitivity: the null 95th percentile in one cell with 3 different seeds (should be similar)
p = [np.percentile(FS.repetir(np.random.SeedSequence(SEED + s, spawn_key=(8, 8, 8, 8)), 300, 738, 0.30, 0.36)["b_R"], 95) for s in (0, 1, 2)]
print("Null P95 of β_mean (n=738, noise 0.30, c=0.36) with 3 seeds:", [round(float(x), 4) for x in p])

## Step 3. Run the whole grid
For each combination of N × noise × coupling the following are run: **null calibration** (2,000), **independent null verification** (1,000) and the worlds with effect 0.05 and 0.10 (1,000 each). It uses all cores. On free Colab it takes about 30 to 60 minutes: leave it running.

In [ ]:
from joblib import Parallel, delayed
def tarea(key, reps, n, noise, c, world, effect, techo, quantile):
    ss = np.random.SeedSequence(SEED, spawn_key=key)
    return key, FS.repetir(ss, reps, n, noise, c, world, effect, techo, quantile)

tasks = []
for i, n in enumerate(N_EL):
    for j, nz in enumerate(NOISES):
        for k, c in enumerate(CS):
            tasks.append(((i, j, k, 0), REPS_CAL, n, nz, c, "null", 0.0, None, False))
            tasks.append(((i, j, k, 1), REPS_VER, n, nz, c, "null", 0.0, None, False))
            for e, eff in enumerate(EFFECTS):
                tasks.append(((i, j, k, 2 + e), REPS_POW, n, nz, c, "linear", eff, None, False))
t0 = time.time()
R_ = dict(Parallel(n_jobs=-1, verbose=2)(delayed(tarea)(*t) for t in tasks))
print("Main grid ready in", round((time.time() - t0) / 60, 1), "min;", len(tasks), "tasks")

## Step 4. Calibrated critical values and expected null slope
The expected null slope **is not zero**: coupling pushes it up, and noise or selection can push it down. That is why the critical value comes from the simulation and not from the normal table. **Critical value (for each N) = the largest null 95th percentile over the whole grid.**

In [ ]:
def p95_se(x, B=200):
    rg = np.random.default_rng(SEED + 99)
    return float(np.std([np.percentile(rg.choice(x, len(x)), 95) for _ in range(B)]))

crit = {}; filas = []
for i, n in enumerate(N_EL):
    for oc in OUT:
        mejor = None
        for j, nz in enumerate(NOISES):
            for k, c in enumerate(CS):
                x = R_[(i, j, k, 0)]["b_" + oc]
                p = float(np.percentile(x, 95))
                filas.append(dict(n=n, ruido=nz, c=c, resultado=oc, pendiente_nula_media=x.mean(), sd=x.std(), p95=p,
                                  elegibles_media=R_[(i, j, k, 0)]["n_pac"].mean() if oc == "R" else R_[(i, j, k, 0)]["n_pac_b"].mean()))
                if mejor is None or p > mejor[0]: mejor = (p, nz, c, j, k)
        crit[(i, oc)] = dict(valor=mejor[0], ruido=mejor[1], c=mejor[2],
                             mc_se=p95_se(R_[(i, mejor[3], mejor[4], 0)]["b_" + oc]))
nulos = pd.DataFrame(filas)
critdf = pd.DataFrame([dict(n=N_EL[i], resultado=oc, valor_critico=v["valor"], celda_que_manda=f"ruido {v['ruido']}, c {v['c']}", error_MC=v["mc_se"])
                       for (i, oc), v in crit.items()])
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 500)
print("CRITICAL VALUES (one-sided, 5 %) and the grid cell that determines them:")
print(critdf.round(4).to_string(index=False))
print("\nEXPECTED NULL SLOPE of β_mean (primary outcome R), mean per cell (n = %d):" % N_EL[-1])
print(nulos[(nulos.resultado == "R") & (nulos.n == N_EL[-1])].pivot(index="c", columns="ruido", values="pendiente_nula_media").round(3))

## Step 5. Verification of the test size (with 1,000 independent null repetitions)

In [ ]:
filas = []
for i, n in enumerate(N_EL):
    for j, nz in enumerate(NOISES):
        for k, c in enumerate(CS):
            v = R_[(i, j, k, 1)]
            for oc in OUT:
                s = float(np.mean(v["b_" + oc] > crit[(i, oc)]["valor"]))
                ingenua = float(np.mean(v["b_" + oc] / v["se_" + oc] > 1.645))      # uncalibrated test (normal, one-sided)
                filas.append(dict(n=n, ruido=nz, c=c, resultado=oc, tamano_calibrado=s, error_MC=np.sqrt(s * (1 - s) / REPS_VER), tamano_sin_calibrar=ingenua))
tam = pd.DataFrame(filas)
print("Test size (target ≤ 0.05). Worst cell per N, for the primary outcome R:")
w = tam[tam.resultado == "R"].sort_values("tamano_calibrado", ascending=False).groupby("n").head(1).sort_values("n")
print(w.round(3).to_string(index=False))
print("\nUNCALIBRATED size (z > 1.645), maximum per N, outcome R: shows why calibration is needed")
print(tam[tam.resultado == "R"].groupby("n").tamano_sin_calibrar.max().round(3).to_string())

## Step 6. Power and sufficiency criteria

In [ ]:
filas = []
for i, n in enumerate(N_EL):
    for j, nz in enumerate(NOISES):
        for k, c in enumerate(CS):
            nul = R_[(i, j, k, 0)]
            for oc in OUT:
                Dbar = {"R": nul["Dbar"].mean(), "Rb": nul["Dbar_b"].mean(), "F": 1.0}[oc]
                dref = 0.10 * Dbar                       # reference effect in the units of the outcome
                semiancho = 1.96 * nul["se_" + oc].mean()
                for e, eff in enumerate(EFFECTS):
                    b = R_[(i, j, k, 2 + e)]["b_" + oc]
                    pw = float(np.mean(b > crit[(i, oc)]["valor"]))
                    filas.append(dict(n=n, ruido=nz, c=c, resultado=oc, efecto_por_SD=eff, poder_calibrado=pw,
                                      error_MC=np.sqrt(pw * (1 - pw) / REPS_POW), pendiente_media_observada=b.mean(),
                                      pendiente_nula_media=nul["b_" + oc].mean(), delta_ref=dref, semiancho_IC=semiancho,
                                      criterio_i_poder=pw >= POWER_TARGET, criterio_ii_precision=semiancho <= dref))
pot = pd.DataFrame(filas)
pot["suficiente"] = pot.criterio_i_poder & pot.criterio_ii_precision
print("CALIBRATED POWER for the reference effect (0.10 per SD), primary outcome R:")
print("(worst grid cell / median / best cell) and how many cells meet BOTH sufficiency criteria")
res = []
for oc in OUT:
    for n in N_EL:
        g = pot[(pot.resultado == oc) & (pot.n == n) & (pot.efecto_por_SD == 0.10)]
        g5 = pot[(pot.resultado == oc) & (pot.n == n) & (pot.efecto_por_SD == 0.05)]
        res.append(dict(resultado=oc, n=n, peor=g.poder_calibrado.min(), mediana=g.poder_calibrado.median(), mejor=g.poder_calibrado.max(),
                        celdas_suficientes=f"{int(g.suficiente.sum())}/{len(g)}", peor_con_efecto_0_05=g5.poder_calibrado.min()))
resumen = pd.DataFrame(res)
print(resumen.round(2).to_string(index=False))
print("\nMonte Carlo error of the power: at most ±%.3f (with %d repetitions)." % (1.96 * 0.5 / np.sqrt(REPS_POW), REPS_POW))

## Step 6b. Grid restricted by the empirical estimate (realistic scenario)
The full grid is the **worst case**: it includes coupling 0 and high noise, which give a very demanding critical value. The protocol says that coupling and noise are **estimated without looking at the outcome** using the prior values and the nadir of the hospital. Here the calculation is repeated with a grid restricted around what was measured in VitalDB (coupling 0.24 to 0.36; noise 0.15 to 0.30) to see how much power is recovered **if** that estimate is confirmed with hospital data.

In [ ]:
C_SUB = [c for c in CS if c >= 0.24]; NZ_SUB = [z for z in NOISES if z <= 0.30]
filas = []
for i, n in enumerate(N_EL):
    for oc in OUT:
        cs_ = max(np.percentile(R_[(i, j, k, 0)]["b_" + oc], 95) for j, z in enumerate(NOISES) if z in NZ_SUB for k, c in enumerate(CS) if c in C_SUB)
        for e, eff in enumerate(EFFECTS):
            pw = [float(np.mean(R_[(i, j, k, 2 + e)]["b_" + oc] > cs_)) for j, z in enumerate(NOISES) if z in NZ_SUB for k, c in enumerate(CS) if c in C_SUB]
            sz = [float(np.mean(R_[(i, j, k, 1)]["b_" + oc] > cs_)) for j, z in enumerate(NOISES) if z in NZ_SUB for k, c in enumerate(CS) if c in C_SUB]
            filas.append(dict(resultado=oc, n=n, efecto_por_SD=eff, valor_critico_acotado=cs_, poder_peor=min(pw), poder_mediano=float(np.median(pw)),
                              tamano_max=max(sz), error_MC_poder=1.96 * 0.5 / np.sqrt(REPS_POW)))
acot = pd.DataFrame(filas)
print("RESTRICTED GRID: coupling", C_SUB, "noise", NZ_SUB)
print(acot[acot.efecto_por_SD == 0.10].round(3).to_string(index=False))
print("\nWith effect 0.05 per SD:")
print(acot[acot.efecto_por_SD == 0.05].round(3).to_string(index=False))

## Step 7. Ceiling contrast (Δβ, secondary) with the same process
It is calibrated with its own null (95th percentile; the largest over the reduced grid) and compared with the average test **in the same ceiling worlds**. Quantile regression is slow, so there are fewer repetitions and a smaller grid here; the Monte Carlo error is shown.

In [ ]:
tasks = []
for i, n in enumerate(N_CEIL):
    for j, nz in enumerate(NOISES_CEIL):
        for k, c in enumerate(CS_CEIL):
            tasks.append(((100 + i, j, k, 4), REPS_CEIL, n, nz, c, "null", 0.0, None, True))
            for e, tc in enumerate(TECHOS):
                tasks.append(((100 + i, j, k, 5 + e), REPS_CEIL, n, nz, c, "ceiling", 0.0, tc, True))
t0 = time.time()
RC = dict(Parallel(n_jobs=-1, verbose=2)(delayed(tarea)(*t) for t in tasks))
print("Ceiling contrast ready in", round((time.time() - t0) / 60, 1), "min")
filas = []
for i, n in enumerate(N_CEIL):
    cdb = max(np.percentile(RC[(100 + i, j, k, 4)]["dbeta_R"], 95) for j in range(len(NOISES_CEIL)) for k in range(len(CS_CEIL)))
    cm = crit[(N_EL.index(n), "R")]["valor"]
    for j, nz in enumerate(NOISES_CEIL):
        for k, c in enumerate(CS_CEIL):
            for e, tc in enumerate(TECHOS):
                r = RC[(100 + i, j, k, 5 + e)]
                pd_ = float(np.mean(r["dbeta_R"] > cdb)); pm = float(np.mean(r["b_R"] > cm))
                filas.append(dict(n=n, ruido=nz, c=c, techo=tc, poder_techo_dbeta=pd_, error_MC_dbeta=np.sqrt(pd_ * (1 - pd_) / REPS_CEIL),
                                  poder_prueba_promedio=pm, error_MC_promedio=np.sqrt(pm * (1 - pm) / REPS_CEIL), valor_critico_dbeta=cdb))
techo = pd.DataFrame(filas)
print(techo.round(3).to_string(index=False))

## Step 8. Figure, files and fingerprints

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for eff, st in ((0.10, "-o"), (0.05, "--s")):
    g = pot[(pot.resultado == "R") & (pot.efecto_por_SD == eff)].groupby("n").poder_calibrado.agg(["min", "median"])
    ax[0].plot(g.index, g["min"], st, label=f"worst cell, effect {eff}/SD"); ax[0].plot(g.index, g["median"], st, alpha=.4, label=f"median, effect {eff}/SD")
ax[0].axhline(0.8, color="k", lw=.8); ax[0].set_xscale("log"); ax[0].set_xlabel("eligible patients"); ax[0].set_ylabel("calibrated power")
ax[0].set_title("Primary test β_mean (outcome R)"); ax[0].legend(fontsize=7)
g = nulos[(nulos.resultado == "R") & (nulos.n == N_EL[-1])]
for nz, gg in g.groupby("ruido"): ax[1].plot(gg.c, gg.pendiente_nula_media, "-o", label=f"noise {nz}")
ax[1].axhline(0, color="k", lw=.8); ax[1].set_xlabel("coupling c"); ax[1].set_ylabel("expected null slope")
ax[1].set_title("Bias of β_mean when there is NO effect"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("sim6_figura.png", dpi=150); plt.close()

nulos.to_csv("sim6_nulo_por_celda.csv", index=False); critdf.to_csv("sim6_valores_criticos.csv", index=False)
tam.to_csv("sim6_tamano_verificacion.csv", index=False); pot.to_csv("sim6_poder_suficiencia.csv", index=False)
resumen.to_csv("sim6_resumen_por_n.csv", index=False); acot.to_csv("sim6_malla_acotada.csv", index=False); techo.to_csv("sim6_techo_dbeta.csv", index=False)
json.dump(dict(fecha=str(datetime.date.today()), semilla=SEED, quick=QUICK, versiones=VERS, N_elegibles=N_EL, ruidos=NOISES, acoplamientos=CS,
               reps=dict(calibracion=REPS_CAL, verificacion=REPS_VER, poder=REPS_POW, techo=REPS_CEIL), efectos=EFFECTS, techos=TECHOS,
               ceiling_grid=dict(N=N_CEIL, ruidos=NOISES_CEIL, c=CS_CEIL), datos_reales="ninguno (todo simulado)"), open("sim6_parametros.json", "w"), indent=2, ensure_ascii=False)
open("requirements_sim6.txt", "w").write("\n".join(f"{k}=={v}" for k, v in VERS.items()) + "\n")
arch = ["sim6_nulo_por_celda.csv", "sim6_valores_criticos.csv", "sim6_tamano_verificacion.csv", "sim6_poder_suficiencia.csv",
        "sim6_resumen_por_n.csv", "sim6_malla_acotada.csv", "sim6_techo_dbeta.csv", "sim6_parametros.json", "requirements_sim6.txt", "sim6_figura.png", "fullsim.py"]
man = {a: hashlib.sha256(open(a, "rb").read()).hexdigest()[:16] for a in arch}
json.dump(man, open("sim6_huellas.json", "w"), indent=2)
print("Files saved (upload ALL of them to GitHub together with this executed notebook):"); [print(" ", a, man.get(a, "")) for a in arch + ["sim6_huellas.json"]]
with __import__("zipfile").ZipFile("sim6_resultados.zip", "w") as z:
    for a in arch + ["sim6_huellas.json"]: z.write(a)
try:
    from google.colab import files; files.download("sim6_resultados.zip")
except Exception as e:
    print("(manual download: sim6_resultados.zip in the files panel)")

## How to read the results
- **Critical values:** the threshold that the estimated slope must exceed to be declared significant (one-sided, 5 %). They come from the worst case of the grid, so they protect against unfavorable coupling or noise.
- **Expected null slope:** what the estimate would be worth **even if there were no effect**. In the final article it is reported next to the estimate.
- **Test size:** it must be ≤ 0.05 (with its Monte Carlo error). The uncalibrated version shows how wrong the usual test would be.
- **Power:** look for ≥ 0.80 in the **worst cell** (conservative reading) and see how many cells meet both criteria.
- **Criterion ii:** the CI half-width must fit within the reference effect.
- **Outcome `F`:** the denominator carries the measurement error of the baseline, so its slope is attenuated and biased downward. It is compared with `R` for the switching rule of v0.5.1 (Section 6.4.7; in v0.6 F is a secondary outcome).
- **Limits:** this is a simulation with the generator's own assumptions; the drop-and-recovery model, the noise, the measurement pattern and the number of events are recomputed with real data (without looking at the outcome) before registration.